In [ ]:
http://github.com/mariamhamza3019-star/Helicobacter-pylori-/blob/main/embedding_generator.py

In [ ]:
!git clone https://github.com/mariamhamza3019-star/Helicobacter-pylori-.git

Cloning into 'Helicobacter-pylori-'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 33 (delta 1), reused 20 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 1.94 MiB | 8.84 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
!pip install -q faiss-cpu sentence-transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 13.4 MB/s eta 0:00:00


In [ ]:
import json
import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

ACG_FILE = "chunks_with_embeddings.json"
# Output files
INDEX_FILE = "h_pylori_faiss.index"
METADATA_FILE = "h_pylori_metadata.json"
RESULTS_FILE = "retrieval_results.csv"
#model
MODEL_NAME = "pritamdeka/S-BioBert-snli-multinli-stsb"

#loading embeddings

In [ ]:
def load_embedding_file(file_path):
    print(f"\n Loading: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "chunks" in data:
        chunks = data["chunks"]
        file_metadata = data.get("metadata", {})
    else:
        chunks = data
        file_metadata = {}

    print(f"Number of chunks: {len(chunks)}")

    if file_metadata:
        print(f"Embedding model: {file_metadata.get('embedding_model', 'N/A')}")
        print(f"Embedding dimension: {file_metadata.get('embedding_dimension', 'N/A')}")

    return chunks, file_metadata


chunks, embedding_metadata = load_embedding_file(ACG_FILE)


 Loading: chunks_with_embeddings.json
Number of chunks: 103
Embedding model: pritamdeka/S-BioBert-snli-multinli-stsb
Embedding dimension: 768


#validate

In [ ]:
missing_embeddings = [
    chunk["chunk_id"]
    for chunk in chunks
    if "embedding" not in chunk
]

if missing_embeddings:
    print(f" {len(missing_embeddings)} chunks are missing embeddings")
    print("Examples:", missing_embeddings[:10])
else:
    print(" All chunks have embeddings")


chunk_ids = [chunk["chunk_id"] for chunk in chunks]

if len(chunk_ids) == len(set(chunk_ids)):
    print("All chunk IDs are unique")
else:
    print(" Duplicate chunk IDs found")

 All chunks have embeddings
All chunk IDs are unique


#embedding matrix

In [ ]:
embeddings = np.array(
    [chunk["embedding"] for chunk in chunks],
    dtype="float32"
)

print("Embedding matrix shape:", embeddings.shape)
print("Number of chunks:", embeddings.shape[0])
print("Vector dimension:", embeddings.shape[1])

Embedding matrix shape: (103, 768)
Number of chunks: 103
Vector dimension: 768


#faiss index

In [ ]:
dimension = embeddings.shape[1]

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Inner Product on normalized vectors = cosine similarity
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)


print(f"Vector dimension : {dimension}")
print(f"Indexed chunks   : {index.ntotal}")
print("Similarity       : Cosine similarity")

Vector dimension : 768
Indexed chunks   : 103
Similarity       : Cosine similarity


In [ ]:
faiss.write_index(index, INDEX_FILE)

INDEX_FILE

'h_pylori_faiss.index'

#save metadata

In [ ]:
metadata = []

for chunk in chunks:
    metadata.append({
        "chunk_id": chunk.get("chunk_id", ""),
        "document_id": chunk.get("document_id", ""),
        "text": chunk.get("text", ""),
        "page": chunk.get("page", chunk.get("page_start", "")),
        "section": chunk.get("section", ""),
        "subsection": chunk.get("subsection", ""),
        "source": chunk.get("source", ""),
        "topic": chunk.get("topic", ""),
        "citation": chunk.get("citation", ""),
        "content_type": chunk.get("content_type", "")
    })


with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

METADATA_FILE

'h_pylori_metadata.json'

In [28]:
print(f"Loading embedding model: {MODEL_NAME}")

model = SentenceTransformer(MODEL_NAME)

print("Model loaded")

Loading embedding model: pritamdeka/S-BioBert-snli-multinli-stsb


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded


#Retrieval function

In [29]:
def retrieve(query, top_k=5):

    # Embed the user's query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query
    faiss.normalize_L2(query_embedding)

    # Search
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        chunk = chunks[idx]

        results.append({
            "chunk_id": chunk.get("chunk_id", ""),
            "score": round(float(score), 4),
            "text": chunk.get("text", ""),
            "document_id": chunk.get("document_id", ""),
            "page": chunk.get("page", chunk.get("page_start", "")),
            "section": chunk.get("section", ""),
            "subsection": chunk.get("subsection", ""),
            "source": chunk.get("source", ""),
            "topic": chunk.get("topic", ""),
            "citation": chunk.get("citation", "")
        })

    return results

#test

In [30]:
query = "What is the recommended treatment for H. pylori infection?"

results = retrieve(query, top_k=5)

for i, result in enumerate(results, start=1):

    print(f"RANK {i}")


    print(f"Chunk ID : {result['chunk_id']}")
    print(f"Score    : {result['score']}")
    print(f"Source   : {result['source']}")
    print(f"Page     : {result['page']}")
    print(f"Section  : {result['section']}")

    print("\nText:")
    print(result["text"])

RANK 1
Chunk ID : ACG_0072
Score    : 0.7101
Source   : ACG Clinical Guideline 2024: Treatment of Helicobacter pylori Infection
Page     : 17
Section  : ERADICATING HELICOBACTER PYLORI INFECTION IN TREATMENT-EXPERIENCED PATIENTS

Text:
5. In treatment-experienced patients with persistent H. pylori infection that is confirmed to be clarithromycin-sensitive, PPI- or PCAB-clarithromycin triple therapy is suggested. As the recommendations of this guideline for the first-line treatment of H. pylori infection are incorporated into clinical practice, a larger proportion of treatment-experienced patients with persistent H. pylori infection will have previously been treated with BQT, rifabutin triple therapy, or vonoprazanamoxicillin dual therapy. If, related to previous use, high cost, or a lack of availability, treatment regimens containing clarithromycin or levofloxacin need to be considered, antibiotic susceptibility testing (see Key concept 6) should be performed. Here, we discuss clarithr

In [31]:
test_queries = [
    "What is the recommended treatment for H. pylori infection?",
    "When should eradication of H. pylori be confirmed?",
    "What are the indications for testing for H. pylori?",
    "What is the recommended duration of H. pylori treatment?",
    "What should be done after H. pylori treatment fails?",
    "What are the recommended first-line therapies for H. pylori?",
    "When should patients be tested for H. pylori after treatment?"
]


all_results = []

for query in test_queries:

    results = retrieve(query, top_k=5)

    for rank, result in enumerate(results, start=1):

        all_results.append({
            "query": query,
            "rank": rank,
            "chunk_id": result["chunk_id"],
            "score": result["score"],
            "source": result["source"],
            "document_id": result["document_id"],
            "page": result["page"],
            "section": result["section"],
            "text": result["text"]
        })


results_df = pd.DataFrame(all_results)

results_df

,query,rank,chunk_id,score,source,document_id,page,section,text
0,What is the recommended treatment for H. pylor...,1,ACG_0072,0.7101,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,17,ERADICATING HELICOBACTER PYLORI INFECTION IN T...,5. In treatment-experienced patients with pers...
1,What is the recommended treatment for H. pylor...,2,ACG_0005,0.7006,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,2,INTRODUCTION,"In some instances, key concepts are based on e..."
2,What is the recommended treatment for H. pylor...,3,ACG_0001,0.6966,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,1,ABSTRACT,"Helicobacter pylori is a prevalent, global inf..."
3,What is the recommended treatment for H. pylor...,4,ACG_0055,0.6889,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,13,POST-TREATMENT TESTING FOR CURE,Results will inform clinicians either to consi...
4,What is the recommended treatment for H. pylor...,5,ACG_0019,0.6867,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,5,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...,"In patients with functional dyspepsia, eradica..."
5,When should eradication of H. pylori be confir...,1,ACG_0029,0.6713,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,6,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...,After successful eradication of H. pylori infe...
6,When should eradication of H. pylori be confir...,2,ACG_0053,0.6610,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,13,POST-TREATMENT TESTING FOR CURE,4. All patients who are treated for H. pylori ...
7,When should eradication of H. pylori be confir...,3,ACG_0055,0.6522,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,13,POST-TREATMENT TESTING FOR CURE,Results will inform clinicians either to consi...
8,When should eradication of H. pylori be confir...,4,ACG_0005,0.6003,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,2,INTRODUCTION,"In some instances, key concepts are based on e..."
9,When should eradication of H. pylori be confir...,5,ACG_0017,0.5994,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,5,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...,2. The determination of when to test for—and t...


In [39]:
# Manual mapping: each of your test_queries -> closest gold question's expect_sections
# (matched by hand from gold_questions.json)
query_expected_sections = {
    "What is the recommended treatment for H. pylori infection?":
        ["ERADICATING HELICOBACTER PYLORI INFECTION IN TREATMENT-NAIVE PATIENTS", "ABSTRACT"],  # ~Q01

    "When should eradication of H. pylori be confirmed?":
        ["POST-TREATMENT TESTING FOR CURE"],  # ~Q12

    "What are the indications for testing for H. pylori?":
        ["INDICATIONS FOR HELICOBACTER PYLORI TESTING AND TREATMENT", "ABSTRACT"],  # ~Q14

    "What is the recommended duration of H. pylori treatment?":
        ["ERADICATING HELICOBACTER PYLORI INFECTION IN TREATMENT-NAIVE PATIENTS", "ABSTRACT"],  # ~Q02

    "What should be done after H. pylori treatment fails?":
        ["ERADICATING HELICOBACTER PYLORI INFECTION IN TREATMENT-EXPERIENCED PATIENTS"],  # ~Q10

    "What are the recommended first-line therapies for H. pylori?":
        ["ERADICATING HELICOBACTER PYLORI INFECTION IN TREATMENT-NAIVE PATIENTS", "ABSTRACT"],  # ~Q01

    "When should patients be tested for H. pylori after treatment?":
        ["POST-TREATMENT TESTING FOR CURE"],  # ~Q12/Q13
}

all_results = []

for query in test_queries:
    results = retrieve(query, top_k=5)
    expected = set(query_expected_sections.get(query, []))
    retrieved_sections = {r["section"] for r in results}
    query_passed = bool(retrieved_sections & expected)  # any overlap = pass

    for rank, result in enumerate(results, start=1):
        chunk_relevant = result["section"] in expected  # per-chunk auto-label

        all_results.append({
            "query": query,
            "rank": rank,
            "chunk_id": result["chunk_id"],
            "score": result["score"],
            "source": result["source"],
            "document_id": result["document_id"],
            "page": result["page"],
            "section": result["section"],
            "text": result["text"],
            "relevant": chunk_relevant,          # <-- auto-filled, no manual reading
            "query_passed": query_passed
        })

results_df = pd.DataFrame(all_results)

# Overall recall@5 across your 7 queries
recall_at_5 = results_df.groupby("query")["query_passed"].first().mean()
print(f"Recall@5: {recall_at_5:.2%}")

results_df

Recall@5: 85.71%


,query,rank,chunk_id,score,source,document_id,page,section,text,relevant,query_passed
0,What is the recommended treatment for H. pylor...,1,ACG_0072,0.7101,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,17,ERADICATING HELICOBACTER PYLORI INFECTION IN T...,5. In treatment-experienced patients with pers...,False,True
1,What is the recommended treatment for H. pylor...,2,ACG_0005,0.7006,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,2,INTRODUCTION,"In some instances, key concepts are based on e...",False,True
2,What is the recommended treatment for H. pylor...,3,ACG_0001,0.6966,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,1,ABSTRACT,"Helicobacter pylori is a prevalent, global inf...",True,True
3,What is the recommended treatment for H. pylor...,4,ACG_0055,0.6889,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,13,POST-TREATMENT TESTING FOR CURE,Results will inform clinicians either to consi...,False,True
4,What is the recommended treatment for H. pylor...,5,ACG_0019,0.6867,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,5,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...,"In patients with functional dyspepsia, eradica...",False,True
5,When should eradication of H. pylori be confir...,1,ACG_0029,0.6713,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,6,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...,After successful eradication of H. pylori infe...,False,True
6,When should eradication of H. pylori be confir...,2,ACG_0053,0.6610,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,13,POST-TREATMENT TESTING FOR CURE,4. All patients who are treated for H. pylori ...,True,True
7,When should eradication of H. pylori be confir...,3,ACG_0055,0.6522,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,13,POST-TREATMENT TESTING FOR CURE,Results will inform clinicians either to consi...,True,True
8,When should eradication of H. pylori be confir...,4,ACG_0005,0.6003,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,2,INTRODUCTION,"In some instances, key concepts are based on e...",False,True
9,When should eradication of H. pylori be confir...,5,ACG_0017,0.5994,ACG Clinical Guideline 2024: Treatment of Heli...,ACG_2024,5,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...,2. The determination of when to test for—and t...,False,True


#saving results

In [36]:
results_df.to_csv(
    RESULTS_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"Retrieval results saved: {RESULTS_FILE}")

Retrieval results saved: retrieval_results.csv


#best result for every query

In [37]:
top_results = results_df[
    results_df["rank"] == 1
][
    [
        "query",
        "chunk_id",
        "score",
        "source",
        "page",
        "section"
    ]
]

top_results

,query,chunk_id,score,source,page,section
0,What is the recommended treatment for H. pylor...,ACG_0072,0.7101,ACG Clinical Guideline 2024: Treatment of Heli...,17,ERADICATING HELICOBACTER PYLORI INFECTION IN T...
5,When should eradication of H. pylori be confir...,ACG_0029,0.6713,ACG Clinical Guideline 2024: Treatment of Heli...,6,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...
10,What are the indications for testing for H. py...,ACG_0021,0.7205,ACG Clinical Guideline 2024: Treatment of Heli...,5,INDICATIONS FOR HELICOBACTER PYLORI TESTING AN...
15,What is the recommended duration of H. pylori ...,ACG_0072,0.6923,ACG Clinical Guideline 2024: Treatment of Heli...,17,ERADICATING HELICOBACTER PYLORI INFECTION IN T...
20,What should be done after H. pylori treatment ...,ACG_0055,0.6944,ACG Clinical Guideline 2024: Treatment of Heli...,13,POST-TREATMENT TESTING FOR CURE
25,What are the recommended first-line therapies ...,ACG_0036,0.7055,ACG Clinical Guideline 2024: Treatment of Heli...,9,ERADICATING HELICOBACTER PYLORI INFECTION IN T...
30,When should patients be tested for H. pylori a...,ACG_0055,0.7538,ACG Clinical Guideline 2024: Treatment of Heli...,13,POST-TREATMENT TESTING FOR CURE


In [38]:
from google.colab import files

files.download(INDEX_FILE)
files.download(METADATA_FILE)
files.download(RESULTS_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>